# 07 — Acquisition-proxy hỗ trợ gợi ý sản phẩm
Dự báo số tháng có acquisition, số sản phẩm mở và điểm acquisition trong **1/3/6 tháng**. Horizon 1 tháng là chính.
Đây là mô hình ưu tiên khách và ước lượng số lượng, chưa dự báo sản phẩm cụ thể hoặc đo MAP@7.

**Input giữ nguyên:** cache sản phẩm và metadata của notebook affinity. Có cả HasEvent/NoEvent.
**Thay đổi:** chọn bằng capture@10% trên validation cho ưu tiên khách, báo riêng model RMSE tốt nhất, baseline constant không có ranking score, xử lý ties công bằng, thêm M-only và RFM-only. Test chỉ xác nhận, không chọn lại model.
Chạy **Restart kernel → Run All**. Kết quả cũ đã xóa để tránh lẫn với tiêu chí mới. `%pip install lifetimes` nếu muốn thử mô hình xác suất và môi trường cho phép cài.

**Theo kế hoạch CLV:** BG/NBD + Gamma-Gamma so với gradient boosting, backtest 6m và sai số theo RFM. Preflight tự cài lifetimes nếu thiếu khi AUTO_INSTALL_LIFETIMES=True. Nếu fit thất bại, báo cáo trạng thái thay vì giả có kết quả. 12m chưa đủ dữ liệu để làm train/validation/test đáng tin.


In [ ]:
from pathlib import Path
from functools import lru_cache
import json, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
ROOT=Path('/kaggle/working' if Path('/kaggle/working').exists() else './clv_work')
# Đổi 2 đường dẫn này nếu output notebook affinity ở nơi khác; None = tìm duy nhất trong /kaggle.
SNAPSHOT_DIR=None
METADATA_DIR=None
OUT=ROOT/'clv_outputs';OUT.mkdir(parents=True,exist_ok=True)
FEATURE_CACHE=OUT/'features';FEATURE_CACHE.mkdir(exist_ok=True)
SEED=42;MAX_TRAIN_ROWS=200000;BATCH_SIZE=50000
RECOMMEND_MONTH='2016-05';DISCOUNT_RATE=0.0
# Validation hoàn tất nhãn trước tháng test đầu tiên. Mỗi horizon có model/selection riêng.
BACKTEST_SPECS={
 1:{'validation':'2015-11','test':['2015-12','2016-01','2016-02','2016-03','2016-04']},
 3:{'validation':'2015-09','test':['2015-12','2016-01','2016-02']}}
EXPORT_HORIZONS=[1,3,6]
ENABLE_BGNBD=True;ENABLE_GAMMA_GAMMA=True
AUTO_INSTALL_LIFETIMES=True # Cài vào kernel hiện tại nếu thiếu. Kaggle cần Internet khi cài.
PROB_PENALIZERS=[.01,.1,1.0] # Thử theo thứ tự cố định, chỉ chọn fit numerically valid, không dựa test score.
if ENABLE_BGNBD:
    import importlib.util, subprocess, sys
    if importlib.util.find_spec('lifetimes') is None:
        if not AUTO_INSTALL_LIFETIMES:
            raise ImportError('Thiếu lifetimes: chạy %pip install lifetimes rồi Restart/Run All. Chưa có benchmark xác suất.')
        print('Cài lifetimes cho kernel hiện tại...')
        try:subprocess.check_call([sys.executable,'-m','pip','install','lifetimes'])
        except subprocess.CalledProcessError as e:
            raise RuntimeError('Không cài được lifetimes. Bật Internet trên Kaggle hoặc cài thư viện trước khi Run All.') from e
    from lifetimes import BetaGeoFitter, GammaGammaFitter
    print('BG/NBD và Gamma-Gamma: thư viện sẵn sàng; fit được kiểm tra trong benchmark.')
# Trọng số là điểm nghiệp vụ, KHÔNG phải monetary thật. Mặc định mọi sản phẩm = 1.
# Chỉ thay bằng bộ trọng số được cố định trước validation; không học weight trên nhãn test.
PRODUCT_WEIGHTS={}
PRODUCTS=['ind_ahor_fin_ult1','ind_aval_fin_ult1','ind_cco_fin_ult1','ind_cder_fin_ult1',
'ind_cno_fin_ult1','ind_ctju_fin_ult1','ind_ctma_fin_ult1','ind_ctop_fin_ult1',
'ind_ctpp_fin_ult1','ind_deco_fin_ult1','ind_deme_fin_ult1','ind_dela_fin_ult1',
'ind_ecue_fin_ult1','ind_fond_fin_ult1','ind_hip_fin_ult1','ind_plan_fin_ult1',
'ind_pres_fin_ult1','ind_reca_fin_ult1','ind_tjcr_fin_ult1','ind_valo_fin_ult1',
'ind_viv_fin_ult1','ind_nomina_ult1','ind_nom_pens_ult1','ind_recibo_ult1']
STATES=['New','Active','At-risk','Churned','Win-back']
TARGETS=['event_months','acquisitions','points']
WEIGHTS=np.array([PRODUCT_WEIGHTS.get(p,1.0) for p in PRODUCTS],float)
assert not (set(PRODUCT_WEIGHTS)-set(PRODUCTS))
assert np.isfinite(WEIGHTS).all() and (WEIGHTS>0).all() and DISCOUNT_RATE>=0

def resolve(configured,name):
    if configured:
        p=Path(configured)
        assert p.is_dir(),f'Không thấy {p}'
        return p
    direct=ROOT/name
    if direct.is_dir():return direct
    candidates=[]
    for base in [Path('/kaggle/input'),ROOT]:
        if base.exists():candidates.extend(p for p in base.rglob(name) if p.is_dir())
    candidates=list(dict.fromkeys(candidates))
    if len(candidates)!=1:raise ValueError(f'{name}: tìm thấy {len(candidates)} folder; đặt đường dẫn thủ công: {candidates}')
    return candidates[0]
SNAPSHOT_DIR=resolve(SNAPSHOT_DIR,'affinity_snapshot_cache')
METADATA_DIR=resolve(METADATA_DIR,'customer_metadata')
MONTHS=sorted(p.stem for p in SNAPSHOT_DIR.glob('????-??.parquet'))
assert MONTHS and RECOMMEND_MONTH in MONTHS
assert all(pd.Period(b,'M')-pd.Period(a,'M')==pd.offsets.MonthEnd(1) for a,b in zip(MONTHS,MONTHS[1:])), 'Cache thiếu tháng'
def shift(m,k):return str(pd.Period(m,'M')+k)
def idx(m):return pd.Period(m,'M').ordinal
print('Snapshot:',SNAPSHOT_DIR,'Metadata:',METADATA_DIR,'Months:',MONTHS[0],MONTHS[-1])
print('Weights:',dict(zip(PRODUCTS,WEIGHTS)),'discount:',DISCOUNT_RATE)

SELECTION_OBJECTIVE='ranking' # ranking: capture10; quantity: RMSE. Chốt trước test.
assert SELECTION_OBJECTIVE in ['ranking','quantity']


## 1. Dữ liệu và đơn vị dự báo
Một dòng = một khách đang có **snapshot sản phẩm đầy đủ** tại tháng chốt. Khách NoEvent vẫn nằm trong cohort. Không dự báo chính cho khách đang vắng mặt; cần model riêng nếu muốn CLV cho At-risk/Churned.
Event được xác định bằng cặp tháng liền kề; quay lại sau gap không được xem toàn bộ danh mục là acquisition.
Nhãn H tháng chỉ xác định khi đủ H cặp tháng tiếp theo. Thiếu một tháng → nhãn thiếu, không gán 0. Vì vậy benchmark có selection bias về khách được quan sát liên tục; luôn xem coverage.

**Phân biệt hai nguồn history:** `ever_acquired` lấy từ F_events của metadata RFM (fallback F_event_months nếu schema thiếu F_events). `has_cached_event`, event_n và bộ BG/NBD được tính từ cache complete-snapshot. Hai nguồn không bị ép bằng nhau; xem history_mismatch_n và event_history_mismatch_YYYY-MM.parquet. Khác biệt có thể do coverage/window/schema, cần kiểm tra trước khi giải thích benchmark. NoEvent của metadata phải tự nhất quán với rfm_status và cụm trống; không xóa cụm khách để vượt assertion.


In [ ]:
@lru_cache(maxsize=2)
def snapshot(m):
    d=pd.read_parquet(SNAPSHOT_DIR/f'{m}.parquet')
    assert d.customer_id.is_unique and d.customer_id.notna().all()
    assert d[PRODUCTS].isin([0,1]).all().all(),f'{m}: cache không đầy đủ/không nhị phân'
    return d.set_index('customer_id')[PRODUCTS].astype('uint8')
@lru_cache(maxsize=2)
def event(m):
    if shift(m,-1) not in MONTHS:return pd.DataFrame(columns=TARGETS,index=pd.Index([],name='customer_id'),dtype=float)
    a,b=snapshot(shift(m,-1)),snapshot(m)
    ids=a.index.intersection(b.index)
    add=(a.loc[ids].to_numpy()==0)&(b.loc[ids].to_numpy()==1)
    n=add.sum(1)
    return pd.DataFrame({'event_months':(n>0).astype(int),'acquisitions':n,'points':add@WEIGHTS},index=ids)

def build_features():
    # Chỉ giữ history aggregate, không nối toàn bộ 13 triệu dòng vào RAM.
    history=pd.DataFrame(columns=['observed_pairs','event_n','product_n','point_n','first_event','last_event','repeat_products','repeat_points'],dtype=float)
    history.index.name='customer_id'
    coverage=[]
    for m in MONTHS:
        current=snapshot(m);ev=event(m)
        history=history.reindex(history.index.union(current.index))
        zeros=['observed_pairs','event_n','product_n','point_n','repeat_products','repeat_points']
        history[zeros]=history[zeros].fillna(0)
        pos=ev.index[ev.event_months.gt(0)]
        repeat=pos[history.loc[pos,'event_n'].gt(0).to_numpy()]
        history.loc[ev.index,'observed_pairs']+=1
        history.loc[ev.index,'event_n']+=ev.event_months.to_numpy()
        history.loc[ev.index,'product_n']+=ev.acquisitions.to_numpy()
        history.loc[ev.index,'point_n']+=ev.points.to_numpy()
        first=pos[history.loc[pos,'first_event'].isna().to_numpy()]
        history.loc[first,'first_event']=idx(m)
        history.loc[pos,'last_event']=idx(m)
        history.loc[repeat,'repeat_products']+=ev.loc[repeat,'acquisitions'].to_numpy()
        history.loc[repeat,'repeat_points']+=ev.loc[repeat,'points'].to_numpy()
        d=history.loc[current.index].copy()
        d['has_cached_event']=d.event_n.gt(0).astype(int)
        d['acquisition_rate']=d.event_n/d.observed_pairs.replace(0,np.nan)
        d['months_since_last_acquisition']=idx(m)-d.last_event
        d['bg_frequency']=(d.event_n-1).clip(lower=0)
        d['bg_recency']=d.last_event-d.first_event
        d['bg_T']=idx(m)-d.first_event
        d['repeat_mean_products']=d.repeat_products/d.bg_frequency.replace(0,np.nan)
        d['repeat_mean_points']=d.repeat_points/d.bg_frequency.replace(0,np.nan)
        for h in [3,6]:
            recent=[event(shift(m,-j)) for j in range(h) if shift(m,-j) in MONTHS]
            recent=[e for e in recent if not e.empty]
            if recent:
                joined=pd.concat(recent,axis=0)
                sums=joined.groupby(level=0)[TARGETS].sum()
                exposures=joined.groupby(level=0).size()
            else:
                sums=pd.DataFrame(0.,index=d.index,columns=TARGETS)
                exposures=pd.Series(0.,index=d.index)
            d[f'valid_pairs_{h}m']=exposures.reindex(d.index).fillna(0)
            for target in TARGETS:d[f'{target}_{h}m']=sums[target].reindex(d.index).fillna(0)
        d['n_products']=current.sum(axis=1);d['remaining_products']=24-d.n_products
        for p in PRODUCTS:d[p]=current[p]
        meta=pd.read_parquet(METADATA_DIR/f'{m}.parquet').set_index('customer_id')
        assert meta.index.is_unique and set(current.index).issubset(meta.index)
        meta=meta.loc[current.index]
        assert meta.is_observed.eq(True).all() and meta.lifecycle.isin(STATES).all()
        for col in ['R','F_event_months','M_group_points','lifecycle','rfm_cluster','rfm_status']:
            assert col in meta,f'Metadata thiếu {col}; dùng output notebook 06 mới'
            d[col]=meta[col]
        # RFM status/F_events là nguồn định nghĩa HasEvent dùng xuyên pipeline.
        # Lịch sử complete-snapshot ở cache có thể khác upstream (khác coverage/window).
        event_col='F_events' if 'F_events' in meta else 'F_event_months'
        upstream_count=pd.to_numeric(meta[event_col],errors='coerce')
        if upstream_count.isna().any() or upstream_count.lt(0).any():
            raise ValueError(f'{m}: {event_col} thiếu/âm; kiểm tra metadata upstream')
        d['ever_acquired']=upstream_count.gt(0).astype(int)
        d['event_history_source']=event_col
        upstream_no_event=meta.rfm_status.eq('NoObservedEvent')
        bad_status=upstream_no_event.ne(d.ever_acquired.eq(0))
        bad_cluster=d.ever_acquired.eq(0)&d.rfm_cluster.notna()
        if bad_status.any() or bad_cluster.any():
            sample=meta.loc[bad_status|bad_cluster,[event_col,'rfm_status','rfm_cluster']].head(20)
            raise ValueError(f'{m}: metadata tự mâu thuẫn HasEvent/NoEvent, không phải lệch cache:\n{sample}')
        d['event_history_mismatch']=d.ever_acquired.ne(d.has_cached_event).astype(int)
        d['cached_months_since_last_acquisition']=d.months_since_last_acquisition
        d['months_since_last_acquisition']=pd.to_numeric(d.R,errors='coerce').where(d.ever_acquired.eq(1))
        mismatch=d.event_history_mismatch.eq(1)
        if mismatch.any():
            audit_cols=['event_n','observed_pairs','has_cached_event','ever_acquired','event_history_source','F_event_months','rfm_status','rfm_cluster']
            d.loc[mismatch,audit_cols].reset_index().to_parquet(OUT/f'event_history_mismatch_{m}.parquet',index=False)
        # Thiếu R không ép bằng 0; HGB xử lý NaN. Lifecycle và NoEvent có cờ riêng.
        for state in STATES:d['life_'+state]=d.lifecycle.eq(state).astype(int)
        d['history_short']=d.observed_pairs.lt(3).astype(int)
        d['month']=m
        d.reset_index().to_parquet(FEATURE_CACHE/f'{m}.parquet',index=False)
        coverage.append({'month':m,'current_complete_n':len(d),'no_event_share':float(d.ever_acquired.eq(0).mean()),'history_pairs_median':float(d.observed_pairs.median()),'cache_no_event_share':float(d.has_cached_event.eq(0).mean()),'history_mismatch_n':int(d.event_history_mismatch.sum())})
    return pd.DataFrame(coverage)
feature_coverage=build_features()
feature_coverage.to_csv(OUT/'feature_coverage.csv',index=False)
display(feature_coverage)
def features(m):return pd.read_parquet(FEATURE_CACHE/f'{m}.parquet')

In [ ]:
label_coverage=[]
def dataset(m,h):
    d=features(m).set_index('customer_id')
    if shift(m,h) not in MONTHS:raise ValueError(f'{m}+{h}: nhãn tương lai chưa đủ')
    y=pd.DataFrame(0.,index=d.index,columns=TARGETS);valid=pd.Series(True,index=d.index)
    for j in range(1,h+1):
        e=event(shift(m,j)).reindex(d.index)
        valid &= e.event_months.notna()
        y['event_months']+=e.event_months.fillna(0)
        y['acquisitions']+=e.acquisitions.fillna(0)
        y['points']+=e.points.fillna(0)/(1+DISCOUNT_RATE)**j
    y.loc[~valid]=np.nan
    label_coverage.append({'month':m,'horizon':h,'origin_n':len(d),'labeled_n':int(valid.sum()),'coverage':float(valid.mean())})
    for target in TARGETS:d['y_'+target]=y[target]
    d['label_end']=shift(m,h)
    return d.loc[valid].reset_index()
# Nhãn points có chiết khấu theo từng tháng; event/product count không chiết khấu.
EDA_MONTHS={1:['2015-11','2015-12','2016-01','2016-02','2016-03','2016-04'],3:['2015-09','2015-12','2016-01','2016-02'],6:['2015-11']}
eda=[]
for h,months in EDA_MONTHS.items():
    for m in months:
        d=dataset(m,h)
        for label,z in d.groupby('ever_acquired'):
            eda.append({'month':m,'horizon':h,'group':'HasEvent' if label else 'NoEvent','n':len(z),
                'future_acquisition_rate':float(z.y_acquisitions.gt(0).mean()),'mean_products':z.y_acquisitions.mean(),'mean_points':z.y_points.mean()})
eda=pd.DataFrame(eda);display(eda.round(4))
pd.DataFrame(label_coverage).drop_duplicates(['month','horizon']).to_csv(OUT/'label_coverage.csv',index=False)
for h in [1,3]:
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for group,z in eda.loc[eda.horizon.eq(h)].groupby('group'):
        axes[0].plot(z.month,z.future_acquisition_rate,marker='o',label=group)
        axes[1].plot(z.month,z.mean_products,marker='o',label=group)
    axes[0].set_title(f'Xác suất có acquisition / {h} tháng')
    axes[1].set_title(f'Số sản phẩm mở trung bình / {h} tháng')
    for ax in axes:ax.legend();ax.tick_params(axis='x',rotation=25)
    plt.tight_layout();plt.show()


## 2. Benchmark và chia thời gian
**1 tháng (benchmark chính):** train origins 03–10/2015 (nhãn hoàn tất ≤11/2015); validation origin 11/2015 → 12/2015; test origins 12/2015–04/2016 → 01–05/2016, cùng tháng chốt với affinity.
**3 tháng:** train origin 04–06/2015 (nhãn hoàn tất ≤09/2015); validation origin 09/2015 (nhãn hoàn tất 12/2015); test origin 12/2015–02/2016. Lựa chọn chỉ dựa validation, không chọn lại theo test. Ở mỗi test cutoff, refit ML bằng các target đã hoàn tất trước/đúng cutoff.
**6 tháng:** origin test 11/2015 → 05/2016; dùng cấu hình ML_full cố định, không tuyên bố là mô hình đã chọn qua validation 6 tháng. Train origins 03–05/2015. Lịch sử ngắn là hạn chế lớn.
**12 tháng:** chưa đủ lịch sử cho đánh giá train/validation/test đáng tin; không xuất benchmark 12 tháng.
RFM bucket không dùng làm ML feature ở train trước tháng fit 11/2015; dùng R/F/M số và phân tích nhóm hybrid tại test.

Horizon 6m vẫn chỉ có một backtest cấu hình cố định. Validation của 1m/3m không được gọi là validation 6m. Có bảng so sánh xác suất với gradient boosting cho từng phase/horizon riêng.


In [ ]:
PORTFOLIO=PRODUCTS+['n_products','remaining_products','observed_pairs','history_short']
STABLE=['ever_acquired','has_cached_event','event_history_mismatch','acquisition_rate','months_since_last_acquisition']+[
    f'{t}_{h}m' for h in [3,6] for t in TARGETS]+['valid_pairs_3m','valid_pairs_6m']
FEATURE_SETS={'ML_M_only':['M_group_points'],'ML_RFM_only':['R','F_event_months','M_group_points'],
 'ML_portfolio':PORTFOLIO,'ML_M':PORTFOLIO+['M_group_points'],
 'ML_RFM':PORTFOLIO+['R','F_event_months','M_group_points'],
 'ML_RFM_stable':PORTFOLIO+['R','F_event_months','M_group_points']+STABLE,
 'ML_full':PORTFOLIO+['R','F_event_months','M_group_points']+STABLE+['life_'+s for s in STATES]}
# Fixed hyperparameters, early_stopping=False: tránh random validation nội bộ customer-month.
MODEL_PARAMS=dict(loss='poisson',max_iter=120,max_leaf_nodes=15,min_samples_leaf=100,
                  learning_rate=.06,l2_regularization=10.,early_stopping=False,random_state=SEED)
fit_status=[]
def training_data(cutoff,h):
    months=[m for m in MONTHS if m>='2015-03' and shift(m,h)<=cutoff]
    assert months, 'Không có nhãn train hoàn tất'
    parts=[]
    per_month=max(1,int(np.ceil(MAX_TRAIN_ROWS/len(months))))
    for m in months:
        part=dataset(m,h)
        if len(part)>per_month:part=part.sample(per_month,random_state=SEED)
        parts.append(part)
    train=pd.concat(parts,ignore_index=True)
    assert train.label_end.le(cutoff).all()
    if len(train)>MAX_TRAIN_ROWS:train=train.sample(MAX_TRAIN_ROWS,random_state=SEED).reset_index(drop=True)
    return train

def fit_models(cutoff,h):
    train=training_data(cutoff,h);models={};global_mean=train[['y_'+t for t in TARGETS]].mean().to_numpy()
    group_mean=train.groupby('ever_acquired')[['y_'+t for t in TARGETS]].mean()
    models['zero']={'kind':'constant','value':np.zeros(3)}
    models['global_mean']={'kind':'constant','value':global_mean}
    models['has_event_mean']={'kind':'group','means':group_mean,'fallback':global_mean}
    for name,cols in FEATURE_SETS.items():
        regressors=[]
        for t in TARGETS:
            y=train['y_'+t].to_numpy()
            if y.sum()==0:regressors.append(0.)
            else:
                reg=HistGradientBoostingRegressor(**MODEL_PARAMS)
                reg.fit(train[cols].astype(float),y);regressors.append(reg)
        models[name]={'kind':'ml','features':cols,'regressors':regressors}
    current=features(cutoff)
    ok=current.ever_acquired.eq(1)&current.bg_T.gt(0)
    eligible=current.loc[ok].copy()
    if ENABLE_BGNBD:
        try:
            from lifetimes import BetaGeoFitter, GammaGammaFitter
            assert len(eligible)>=20 and eligible.bg_frequency.gt(0).sum()>=5, 'Quá ít khách/event lặp lại để fit'
            # frequency là số lần LẶP LẠI, recency là last-first, khác RFM R.
            collapsed=eligible.groupby(['bg_frequency','bg_recency','bg_T']).size().reset_index(name='weight')
            bg=None;errors=[];used_penalty=None
            for penalty in PROB_PENALIZERS:
                try:
                    candidate=BetaGeoFitter(penalizer_coef=penalty)
                    candidate.fit(collapsed.bg_frequency,collapsed.bg_recency,collapsed.bg_T,weights=collapsed.weight)
                    check=np.column_stack([np.asarray(candidate.conditional_expected_number_of_purchases_up_to_time(j,eligible.bg_frequency,eligible.bg_recency,eligible.bg_T)) for j in range(1,h+1)])
                    assert np.isfinite(check).all() and (check>=0).all(), 'Prediction BG không hữu hạn/âm'
                    assert (np.diff(check,axis=1)>=-1e-8).all(), 'Expected cumulative events giảm theo horizon'
                    bg=candidate;used_penalty=penalty;break
                except Exception as e:errors.append(f'penalty={penalty}: {str(e)[:300]}')
            if bg is None:raise RuntimeError('BG/NBD không fit hợp lệ: '+'; '.join(errors))
            rep=train.loc[train.bg_frequency.gt(0)]
            mean_products=float(rep.repeat_products.sum()/max(rep.bg_frequency.sum(),1))
            mean_points=float(rep.repeat_points.sum()/max(rep.bg_frequency.sum(),1))
            models['BGNBD']={'kind':'bg','bg':bg,'product_value':mean_products,'point_value':mean_points,
                'fallback':models['has_event_mean']}
            fit_status.append({'cutoff':cutoff,'horizon':h,'model':'BGNBD','status':'fit','eligible_n':len(eligible),'penalizer':used_penalty,'current_n':len(current),'fallback_share':1-len(eligible)/max(len(current),1)})
            if ENABLE_GAMMA_GAMMA:
                money=eligible.loc[eligible.bg_frequency.gt(0)&eligible.repeat_mean_points.gt(0)]
                corr=money.bg_frequency.corr(money.repeat_mean_points,method='spearman')
                try:
                    assert len(money)>=20, 'Quá ít event lặp lại cho Gamma-Gamma'
                    gg_collapsed=money.groupby(['bg_frequency','repeat_mean_points']).size().reset_index(name='weight')
                    gg=None;gg_errors=[];gg_penalty=None
                    for penalty in PROB_PENALIZERS:
                        try:
                            candidate=GammaGammaFitter(penalizer_coef=penalty)
                            candidate.fit(gg_collapsed.bg_frequency,gg_collapsed.repeat_mean_points,weights=gg_collapsed.weight,q_constraint=True)
                            values=np.asarray(candidate.conditional_expected_average_profit(money.bg_frequency,money.repeat_mean_points))
                            assert np.isfinite(values).all() and (values>0).all(), 'Gamma-Gamma prediction không hợp lệ'
                            gg=candidate;gg_penalty=penalty;break
                        except Exception as e:gg_errors.append(f'penalty={penalty}: {str(e)[:300]}')
                    if gg is None:raise RuntimeError('Gamma-Gamma không fit hợp lệ: '+'; '.join(gg_errors))
                    models['BGNBD_GG']={**models['BGNBD'],'gg':gg}
                    fit_status.append({'cutoff':cutoff,'horizon':h,'model':'BGNBD_GG','status':'experimental_fit','penalizer':gg_penalty,'repeat_value_n':len(money),'frequency_value_spearman':corr})
                except Exception as e:
                    fit_status.append({'cutoff':cutoff,'horizon':h,'model':'BGNBD_GG','status':'skipped','reason':str(e)})
        except ImportError:
            fit_status.append({'cutoff':cutoff,'horizon':h,'model':'BGNBD','status':'missing_lifetimes'})
        except Exception as e:
            fit_status.append({'cutoff':cutoff,'horizon':h,'model':'BGNBD','status':'failed','reason':str(e)})
            if ENABLE_GAMMA_GAMMA:fit_status.append({'cutoff':cutoff,'horizon':h,'model':'BGNBD_GG','status':'blocked_by_BG_failure','reason':str(e)})
    return models

def predict(model,d,h):
    if model['kind']=='constant':p=np.tile(model['value'],(len(d),1))
    elif model['kind']=='group':
        p=np.tile(model['fallback'],(len(d),1))
        for group,values in model['means'].iterrows():p[d.ever_acquired.eq(group).to_numpy()]=values.to_numpy()
    elif model['kind']=='ml':
        p=np.column_stack([np.full(len(d),r) if isinstance(r,float) else r.predict(d[model['features']].astype(float)) for r in model['regressors']])
    else:
        p=predict(model['fallback'],d,h)
        ok=d.ever_acquired.eq(1)&d.bg_T.gt(0)
        z=d.loc[ok]
        if len(z):
            cumulative=np.column_stack([np.asarray(model['bg'].conditional_expected_number_of_purchases_up_to_time(j,z.bg_frequency,z.bg_recency,z.bg_T)) for j in range(1,h+1)])
            assert np.isfinite(cumulative).all(), 'BG/NBD prediction không hữu hạn'
            increments=np.diff(np.column_stack([np.zeros(len(z)),cumulative]),axis=1).clip(min=0)
            expected= increments.sum(1)
            point_value=np.full(len(z),model['point_value'])
            if 'gg' in model:
                repeat=z.bg_frequency.gt(0)&z.repeat_mean_points.gt(0)
                point_value[repeat]=np.asarray(model['gg'].conditional_expected_average_profit(z.loc[repeat,'bg_frequency'],z.loc[repeat,'repeat_mean_points']))
            p[ok,0]=expected;p[ok,1]=expected*model['product_value']
            p[ok,2]=(increments/(1+DISCOUNT_RATE)**np.arange(1,h+1)).sum(1)*point_value
    assert np.isfinite(p).all() and (p>=0).all(), 'Prediction không hợp lệ'
    return p


In [ ]:
def metrics_for(y,p):
    y=np.asarray(y,float);p=np.asarray(p,float)
    assert len(y)>0 and np.isfinite(y).all() and np.isfinite(p).all()
    k=max(1,int(np.ceil(.1*len(y))));total=y.sum();base=y.mean()
    informative=np.unique(p).size>1
    capture=lift=np.nan
    if informative and total>0:
        # Các khách cùng điểm ở ranh giới nhận fractional inclusion như random tie-break.
        threshold=np.partition(p,len(p)-k)[len(p)-k]
        high=p>threshold;ties=p==threshold
        remaining=k-int(high.sum())
        selected_actual=y[high].sum()+y[ties].sum()*remaining/max(int(ties.sum()),1)
        capture=selected_actual/total;lift=(selected_actual/k)/base
    return dict(n=len(y),MAE=mean_absolute_error(y,p),RMSE=np.sqrt(mean_squared_error(y,p)),
        actual_mean=base,predicted_mean=p.mean(),calibration_ratio=p.sum()/total if total>0 else np.nan,
        capture10=capture,lift10=lift,ranking_informative=informative,
        random_capture_reference=k/len(y),random_lift_reference=1.)

def evaluate_models(models,d,m,h,phase):
    rows=[];group_rows=[];predictions={}
    for name,model in models.items():
        parts=[predict(model,d.iloc[start:start+BATCH_SIZE],h) for start in range(0,len(d),BATCH_SIZE)]
        p=np.vstack(parts);predictions[name]=p
        for j,t in enumerate(TARGETS):
            rows.append(dict(phase=phase,month=m,horizon=h,method=name,target=t,**metrics_for(d['y_'+t],p[:,j]),
                             event_prediction_over_H_share=float((p[:,0]>h).mean())))
            for col in ['ever_acquired','lifecycle','rfm_cluster','history_short']:
                for label,ix in d.groupby(col).indices.items():
                    group_rows.append(dict(phase=phase,month=m,horizon=h,method=name,target=t,group_type=col,group=str(label),
                                           **metrics_for(d.iloc[ix]['y_'+t],p[ix,j])))
    return pd.DataFrame(rows),pd.DataFrame(group_rows),predictions
SELECTED_BY_H={};RANKING_SELECTED={};QUANTITY_SELECTED={};selection_rows=[];validation_parts=[];validation_group_parts=[]
for h,spec in BACKTEST_SPECS.items():
    m=spec['validation']
    assert shift(m,h)<=min(spec['test']), 'Validation nhãn chưa biết tại test'
    models=fit_models(m,h);d=dataset(m,h)
    a,b,_=evaluate_models(models,d,m,h,'validation')
    RANKING_SELECTED[h]={};QUANTITY_SELECTED[h]={}
    for t in TARGETS:
        candidates=a.loc[a.target.eq(t)]
        ranked=candidates.loc[candidates.ranking_informative & candidates.capture10.notna()]
        assert len(ranked), f'{h}m/{t}: không có model phân biệt được khách trên validation'
        ranking=ranked.sort_values(['capture10','RMSE','method'],ascending=[False,True,True]).iloc[0]
        quantity=candidates.sort_values(['RMSE','method']).iloc[0]
        RANKING_SELECTED[h][t]=ranking['method'];QUANTITY_SELECTED[h][t]=quantity['method']
        selection_rows.append({'horizon':h,'target':t,'ranking_method':ranking['method'],
            'validation_capture10':ranking.capture10,'quantity_method':quantity['method'],'validation_RMSE':quantity.RMSE})
    SELECTED_BY_H[h]=(RANKING_SELECTED if SELECTION_OBJECTIVE=='ranking' else QUANTITY_SELECTED)[h].copy()
    validation_parts.append(a);validation_group_parts.append(b)
val=pd.concat(validation_parts,ignore_index=True)
val_groups=pd.concat(validation_group_parts,ignore_index=True)
val.to_csv(OUT/'validation_metrics.csv',index=False)
val_groups.to_csv(OUT/'validation_groups.csv',index=False)
(OUT/'selection.json').write_text(json.dumps({'selected_by_horizon':SELECTED_BY_H,'backtest_specs':BACKTEST_SPECS,'objective':SELECTION_OBJECTIVE,'ranking_selected':RANKING_SELECTED,'quantity_selected':QUANTITY_SELECTED,'criterion':'capture10 then RMSE for ranking; RMSE for quantity; validation only'},indent=2))
selection_table=pd.DataFrame(selection_rows)
selection_table.to_csv(OUT/'validation_selection.csv',index=False)
display(selection_table.round(4))
for (h,t),z in val.groupby(['horizon','target']):
    print(f'VALIDATION {h}m / {t}: xếp theo capture10, baseline constant không có ranking')
    with pd.option_context('display.max_rows',None,'display.max_columns',None,'display.width',180):
        display(z[['method','RMSE','MAE','capture10','lift10','calibration_ratio','ranking_informative']]
                .sort_values(['capture10','RMSE'],ascending=[False,True],na_position='last').round(4))
print('Khóa trước test, mục tiêu:',SELECTION_OBJECTIVE,'; phương pháp:',SELECTED_BY_H)


**BG/NBD là benchmark thử nghiệm:** event-month là thời gian rời rạc, khách có thể reopen và dữ liệu bị left truncation. Khách NoEvent hoặc mới có event ở cutoff (T=0) dùng baseline HasEvent/NoEvent; không ép NoEvent thành frequency=0 của BG/NBD. Đếm repeat event-month của BG/NBD KHÁC số product acquisitions; chuyển sang product/points bằng giá trị trung bình một event là giả định cần kiểm chứng.
Gamma-Gamma dùng repeat-event acquisition points; xem Spearman frequency/value và sai số ngoài mẫu. Corr gần 0 không chứng minh độc lập. Mặc định trọng số bằng nhau nên points chính là acquisitions nếu discount=0; không gọi kết quả này là monetary thật.
BG/NBD output là **số event kỳ vọng**, không dùng như xác suất mở ≥1 sản phẩm. Không cắt dự báo BG/NBD về H để che sai giả định; báo cáo tỷ lệ vượt H.

**Monetary proxy:** Gamma-Gamma dự báo điểm trung bình một event-month lặp lại. Khi weight=1, điểm là số sản phẩm mở trong event-month. BGNBD_GG khác BGNBD ở dự báo points, không ở event_months/acquisitions. Vì không có giá giao dịch, đây chỉ là thử nghiệm thống kê, không diễn giải thành lợi nhuận. Không dùng renta hoặc M hiện tại làm giá trị một event.


In [ ]:
all_metrics=[];all_groups=[];selected_test=[];test_prediction_details=[]
for h,spec in BACKTEST_SPECS.items():
    choices=SELECTED_BY_H[h]
    for m in spec['test']:
        models=fit_models(m,h);d=dataset(m,h)
        missing=set(choices.values())-set(models)
        if missing:raise RuntimeError(f'{m}/{h}m: phương pháp đã chọn không fit được: {missing}; xem fit_status')
        a,b,ps=evaluate_models(models,d,m,h,'test');all_metrics.append(a);all_groups.append(b)
        for j,t in enumerate(TARGETS):
            z=d[['customer_id','month','ever_acquired','lifecycle','rfm_cluster','history_short','y_'+t]].copy()
            z['prediction']=ps[choices[t]][:,j];z['target']=t;z['method']=choices[t];z['horizon']=h
            z=z.rename(columns={'y_'+t:'actual'})
            z.to_parquet(OUT/f'test_predictions_{h}m_{m}_{t}.parquet',index=False)
            test_prediction_details.append(z)
            selected_test.append(a.loc[a.target.eq(t)&a.method.eq(choices[t])])
        print('Test hoàn tất:',m,'horizon:',h)
# 6m fixed benchmark, không chọn model từ bảng này.
models6=fit_models('2015-11',6);d6=dataset('2015-11',6)
a,b,_=evaluate_models(models6,d6,'2015-11',6,'fixed_6m_backtest');all_metrics.append(a);all_groups.append(b)
metrics=pd.concat(all_metrics,ignore_index=True);group_metrics=pd.concat(all_groups,ignore_index=True)
metrics.to_csv(OUT/'test_metrics.csv',index=False);group_metrics.to_csv(OUT/'test_groups.csv',index=False)
pd.DataFrame(fit_status).to_csv(OUT/'fit_status.csv',index=False)
pd.DataFrame(label_coverage).drop_duplicates(['month','horizon']).to_csv(OUT/'label_coverage.csv',index=False)
summary=metrics.groupby(['phase','horizon','target','method'])[['MAE','RMSE','capture10','lift10','calibration_ratio']].mean()
for (phase,h,t),z in summary.reset_index().groupby(['phase','horizon','target']):
    print(f'{phase.upper()} {h}m / {t}: trung bình metric từng tháng, không chọn lại model')
    with pd.option_context('display.max_rows',None,'display.max_columns',None,'display.width',180):
        display(z[['method','RMSE','MAE','capture10','lift10','calibration_ratio']]
                .sort_values(['capture10','RMSE'],ascending=[False,True],na_position='last').round(4))
selected_test_metrics=pd.concat(selected_test,ignore_index=True)
selected_test_metrics.to_csv(OUT/'selected_test_metrics.csv',index=False)
print('PHƯƠNG PHÁP KHÓA TỪ VALIDATION: kết quả từng tháng')
with pd.option_context('display.max_rows',None,'display.max_columns',None):
    display(selected_test_metrics[['horizon','month','target','method','RMSE','capture10','lift10','calibration_ratio']].round(4))
status=pd.DataFrame(fit_status)
if len(status):
    display(status)
    if status.status.eq('missing_lifetimes').any():print('BG/NBD chưa train: thiếu lifetimes. Cài rồi Run All nếu muốn so sánh.')


# Benchmark đúng các họ mô hình trong kế hoạch, tách riêng với feature ablation.
family_methods=['BGNBD','BGNBD_GG','ML_RFM','ML_RFM_stable','ML_full','has_event_mean']
family_comparison=pd.concat([val,metrics],ignore_index=True)
family_comparison=family_comparison.loc[family_comparison.method.isin(family_methods)]
family_comparison.to_csv(OUT/'probabilistic_vs_gradient_boosting.csv',index=False)
print('BG/NBD vs Gamma-Gamma vs gradient boosting: target points thể hiện phần value model')
for (phase,h),z in family_comparison.loc[family_comparison.target.eq('points')].groupby(['phase','horizon']):
    print(phase,h,'tháng')
    with pd.option_context('display.max_rows',None,'display.max_columns',None):
        display(z[['month','method','n','RMSE','capture10','calibration_ratio']].round(4))
coverage_models=[]
for phase,h,m in [('validation',h,spec['validation']) for h,spec in BACKTEST_SPECS.items()]+[('test',h,m) for h,spec in BACKTEST_SPECS.items() for m in spec['test']]+[('fixed_6m_backtest',6,'2015-11')]:
    for method in ['BGNBD','BGNBD_GG']:
        fit_rows=[r for r in fit_status if r['cutoff']==m and r['horizon']==h and r['model']==method]
        row=fit_rows[-1] if fit_rows else {'status':'disabled_or_not_attempted'}
        evaluated=((family_comparison.phase==phase)&(family_comparison.horizon==h)&(family_comparison.month==m)&(family_comparison.method==method)).any()
        coverage_models.append({'phase':phase,'horizon':h,'month':m,'method':method,'fit_status':row['status'],'evaluated':bool(evaluated),'reason':row.get('reason','')})
probability_status=pd.DataFrame(coverage_models)
probability_status.to_csv(OUT/'probabilistic_benchmark_status.csv',index=False)
display(probability_status)
if not probability_status.evaluated.all():
    print('CHƯA HOÀN THÀNH toàn bộ benchmark xác suất. Xem status/reason; không kết luận ML thắng các model chưa fit.')
else:print('Các mô hình xác suất đã được đánh giá ở tất cả cutoff cấu hình.')


## 3. Đánh giá mô hình đã khóa
Capture/lift kiểm tra ưu tiên khách, RMSE kiểm tra mức dự báo, calibration kiểm tra tổng dự báo/thực tế. MAE giữ làm diagnostic, không chọn ranking model.
Baseline constant có capture/lift = NaN vì không phân biệt được khách. Đường capture 10% biểu thị chọn ngẫu nhiên. Với tie ở ranh giới, metric dùng kỳ vọng random tie-break thay vì lợi dụng thứ tự ID.
Biểu đồ sau chỉ dùng phương pháp khóa trên validation. Nhãn các tháng test có thể chồng nhau, không diễn giải các tháng như thí nghiệm độc lập.


In [ ]:
selected_metrics=pd.concat(selected_test,ignore_index=True)
cal=[]
for h in BACKTEST_SPECS:
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    for t,z in selected_metrics.loc[selected_metrics.horizon.eq(h)].groupby('target'):
        axes[0].plot(z.month,z.MAE,marker='o',label=t)
        axes[1].plot(z.month,z.calibration_ratio,marker='o',label=t)
        axes[2].plot(z.month,z.capture10,marker='o',label=t)
    axes[0].set_title(f'MAE từng tháng / {h}m')
    axes[1].set_title(f'Tổng dự báo / thực tế / {h}m');axes[1].axhline(1,color='gray',ls='--')
    axes[2].set_title(f'Capture top10% / {h}m');axes[2].axhline(.1,color='gray',ls='--')
    for ax in axes:ax.legend();ax.tick_params(axis='x',rotation=25)
    plt.tight_layout();plt.show()
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    for ax,t in zip(axes,TARGETS):
        z=pd.concat([d for d in test_prediction_details if d.target.iloc[0]==t and d.horizon.iloc[0]==h],ignore_index=True)
        z['decile']=pd.qcut(z.prediction,10,duplicates='drop') if z.prediction.nunique()>1 else 'All_equal'
        table=z.groupby('decile',observed=True).agg(n=('actual','size'),predicted=('prediction','mean'),actual=('actual','mean')).reset_index()
        table['target']=t;table['horizon']=h;cal.append(table)
        ax.plot(range(len(table)),table.predicted,marker='o',label='Dự báo')
        ax.plot(range(len(table)),table.actual,marker='o',label='Thực tế')
        ax.set_title(f'{t} / {h}m');ax.set_xlabel('Bucket tăng theo prediction');ax.legend()
    plt.tight_layout();plt.show()
pd.concat(cal,ignore_index=True).to_csv(OUT/'test_calibration.csv',index=False)
sel_group=group_metrics.loc[group_metrics.apply(lambda r:r.horizon in SELECTED_BY_H and r.method==SELECTED_BY_H.get(r.horizon,{}).get(r.target),axis=1)]
# Bảng đọc nhanh: mỗi horizon/target tách riêng, chỉ HasEvent/NoEvent và lifecycle.
for h in BACKTEST_SPECS:
    for group_type in ['ever_acquired','lifecycle']:
        z=sel_group.loc[sel_group.horizon.eq(h)&sel_group.target.eq('acquisitions')&sel_group.group_type.eq(group_type)]
        print(f'SAI SỐ THEO NHÓM {h}m / acquisitions / {group_type}')
        with pd.option_context('display.max_rows',None,'display.max_columns',None):
            display(z[['month','group','n','RMSE','capture10','calibration_ratio']].round(4))
sel_group.to_csv(OUT/'selected_test_groups.csv',index=False)

# Sai số theo RFM: gộp các dòng customer-month test, không phải khách duy nhất.
rfm_rows=sel_group.loc[sel_group.horizon.eq(1)&sel_group.target.eq('acquisitions')&sel_group.group_type.eq('rfm_cluster')].copy()
rfm_report=[]
for label,z in rfm_rows.groupby('group'):
    count=z.n.sum();actual=(z.n*z.actual_mean).sum();predicted=(z.n*z.predicted_mean).sum()
    rfm_report.append({'rfm_cluster':label,'customer_month_n':count,'actual_mean':actual/count,
        'predicted_mean':predicted/count,'RMSE':np.sqrt((z.n*z.RMSE**2).sum()/count),
        'calibration_ratio':predicted/actual if actual>0 else np.nan})
rfm_error=pd.DataFrame(rfm_report)
if len(rfm_error):
    rfm_error=rfm_error.sort_values('customer_month_n',ascending=False)
    rfm_error.to_csv(OUT/'rfm_error_1m.csv',index=False)
    print('Sai số acquisition 1 tháng theo cụm RFM, model khóa validation')
    display(rfm_error.head(12).round(4))
    top=rfm_error.head(8);positions=np.arange(len(top))
    fig,axes=plt.subplots(1,2,figsize=(14,4))
    axes[0].bar(positions-.18,top.actual_mean,width=.36,label='Thực tế')
    axes[0].bar(positions+.18,top.predicted_mean,width=.36,label='Dự báo')
    axes[0].set_title('Số acquisition trung bình');axes[0].legend()
    axes[1].bar(positions,top.RMSE);axes[1].set_title('RMSE theo cụm')
    for ax in axes:ax.set_xticks(positions,top.rfm_cluster,rotation=40,ha='right')
    plt.tight_layout();plt.show()
else:print('Không có RFM cluster đủ nhãn để phân tích; xem metadata và rfm_status.')
print('NoEvent không bị bỏ khỏi benchmark: xem bảng ever_acquired riêng vì nhóm này không có cụm RFM.')


In [ ]:
# Feature drift: so sánh train-vs-cutoff bằng descriptive stats, không tune trên test.
drift=[]
for h,spec in BACKTEST_SPECS.items():
  for m in spec['test']:
    train=training_data(m,h);current=features(m)
    for col in ['R','F_event_months','M_group_points','observed_pairs','acquisition_rate','event_months_3m','n_products']:
        for label,z in [('train',train),('current',current)]:
            x=pd.to_numeric(z[col],errors='coerce')
            drift.append({'horizon':h,'month':m,'sample':label,'feature':col,'missing_share':x.isna().mean(),'median':x.median(),'p95':x.quantile(.95),'max':x.max()})
drift=pd.DataFrame(drift);drift.to_csv(OUT/'feature_drift.csv',index=False)
display(drift.round(4))
print('Target drift: xem actual_mean trong test_metrics; feature ổn định không bảo đảm target ổn định.')


## 4. Dự báo 05/2016 và liên kết affinity
Export 1 và 3 tháng bằng phương pháp khóa từ validation; 6 tháng bằng ML_full cố định. Đây là dự báo tương lai chưa có nhãn, không gọi là test.
Chỉ export khách có current complete snapshot; lưu coverage và rõ khách excluded. Không suy diễn NoCurrentSnapshot = NoEvent. At-risk/Churned cần mô hình/đặc tả bổ sung trước khi dự báo cho toàn bộ known customers.
CLV-proxy dùng để ưu tiên khách, affinity dùng để xếp sản phẩm. Join bằng customer_id + as_of_month; không nhân trực tiếp ranking_score của affinity với CLV rồi gọi là expected revenue vì ranking_score chưa là xác suất đã calibration.


In [ ]:
exports=[]
current=features(RECOMMEND_MONTH)
for h in EXPORT_HORIZONS:
    models=fit_models(RECOMMEND_MONTH,h)
    choices=SELECTED_BY_H[h] if h in SELECTED_BY_H else {t:'ML_full' for t in TARGETS}
    assert set(choices.values()).issubset(models), 'Selected model không fit được; xem fit_status'
    out=current[['customer_id','month','lifecycle','rfm_cluster','rfm_status','ever_acquired']].copy().rename(columns={'month':'as_of_month'})
    out['horizon_months']=h;out['prediction_status']='current_complete_snapshot'
    for j,t in enumerate(TARGETS):
        name=choices[t]
        out['predicted_'+t]=np.concatenate([predict(models[name],current.iloc[start:start+BATCH_SIZE],h)[:,j] for start in range(0,len(current),BATCH_SIZE)])
        out['method_'+t]=name
    assert out.customer_id.is_unique
    out.to_parquet(OUT/f'clv_predictions_{RECOMMEND_MONTH}_{h}m.parquet',index=False)
    with (OUT/f'models_{RECOMMEND_MONTH}_{h}m.pkl').open('wb') as f:pickle.dump({name:models[name] for name in set(choices.values())},f)
    exports.append(out)
known=pd.read_parquet(METADATA_DIR/f'{RECOMMEND_MONTH}.parquet')
status=known[['customer_id','is_observed','lifecycle']].copy()
status['clv_status']=np.where(status.customer_id.isin(current.customer_id),'predicted_current_complete',np.where(status.is_observed,'excluded_incomplete_snapshot','excluded_no_current_snapshot'))
status.to_parquet(OUT/f'prediction_coverage_{RECOMMEND_MONTH}.parquet',index=False)
display(status.groupby(['clv_status','lifecycle']).size().rename('n').reset_index())
display(exports[0].sort_values('predicted_points',ascending=False).head(10))
pd.DataFrame(fit_status).to_csv(OUT/'fit_status.csv',index=False)
config={'snapshot_dir':str(SNAPSHOT_DIR),'metadata_dir':str(METADATA_DIR),'weights':dict(zip(PRODUCTS,WEIGHTS)),
 'discount_rate':DISCOUNT_RATE,'selected_by_horizon':SELECTED_BY_H,'selection_objective':SELECTION_OBJECTIVE,'backtest_specs':BACKTEST_SPECS,'fixed_6m':'ML_full','model_params':MODEL_PARAMS,'seed':SEED}
(OUT/'config.json').write_text(json.dumps(config,indent=2))
feature_spec={'key':['customer_id','as_of_month'],'version':'acquisition_proxy_v4_selection','refresh':'monthly',
 'ttl':'until_next_monthly_snapshot','targets':TARGETS,'horizons':EXPORT_HORIZONS,
 'universe':'customers with complete current product snapshot; NoEvent included',
 'label_policy':'all H consecutive future pairs complete, else excluded and coverage reported',
 'value_unit':'business acquisition points, not currency','weights':dict(zip(PRODUCTS,WEIGHTS)),
 'features':FEATURE_SETS,'NoEvent':'metadata F_events (fallback F_event_months) equals 0; RFM cluster null; BG uses documented fallback',
 'history_sources':'upstream HasEvent and complete-snapshot BG history audited separately',
 'limitations':['left truncation','monthly discretization','reopen counts as acquisition','finite product catalog','label observation selection bias','few independent long-horizon test periods']}
(OUT/'feature_spec.json').write_text(json.dumps(feature_spec,indent=2))
print('Outputs:',OUT)


## Kết luận cần ghi sau Run All
1. M-only và RFM-only khác nhau thế nào ngoài mẫu? Portfolio thêm thông tin gì?
2. Model khóa bằng validation có duy trì capture và calibration theo tháng test không?
3. Stable features và lifecycle có thêm lợi ích so với RFM không?
4. HasEvent/NoEvent, New và lịch sử ngắn có nhóm nào dự báo sai mức?
5. BG/NBD đã fit chưa? Nếu missing/failed thì chưa có bằng chứng so sánh mô hình đó.

Acquisition-proxy dự báo số lượng và mức ưu tiên khách. Để dự đoán sản phẩm cụ thể, cần affinity/model từng sản phẩm và MAP@7. Nhân cùng một điểm khách vào mọi điểm sản phẩm không đổi ranking.
Model selection thực hiện trước test. Bảng test chỉ xác nhận, không thay đổi selection theo bảng xếp hạng test.
